In [16]:
from qiskit.quantum_info import SparsePauliOp
from sympy import Matrix, latex, nsimplify, Float
from IPython.display import display, Latex
import numpy as np

## My own expression:
$$
\begin{align}
H = &, h_{00}\left[\frac{1}{2}(IIII - ZIII)\right] +  h_{11}\left[\frac{1}{2}(IIII - ZZII)\right]+  h_{22}\left[\frac{1}{2}(IIII - IIZI)\right] +  h_{33}\left[\frac{1}{2}(IIII - IZZZ)\right] \nonumber \\
&+ h_{0110}\left[\frac{1}{4}(IIII - ZIII - ZZII + IZII)\right]
+ h_{2332}\left[\frac{1}{4}(IIII - IIZI - IZZZ + IZIZ)\right] \nonumber \\
&+: h_{0330}\left[\frac{1}{4}(IIII - ZIII - IZZZ + ZZZZ)\right]  +h_{1221} \left[\frac{1}{4}(IIII - IIZI - ZZII + ZZZI)\right] \nonumber \\
&+ (h_{0220} - h_{0202})\left[\frac{1}{4}(IIII - IIZI - ZIII + ZIZI)\right] + (h_{1331} - h_{1313})\left[\frac{1}{4}(IIII - IZZZ - ZZII + ZIZZ)\right] \nonumber \\
&+h_{0132}\left[\frac{1}{8}(XIXI + XZXI + YIYI + YZYI + XIXZ + XZXZ + YIYZ + YZYZ)\right] \nonumber \\
&+ h_{0312}\left[\frac{1}{8}(-XIXI + XZXI - YIYI + YZYI - XIXZ + XZXZ - YIYZ + YZYZ)\right] \\
\end{align}
$$

Which, after some manipulation becomes
$$
\begin{align}
H = &\mathbf{IIII} \left[\frac{1}{2}(h_{00} + h_{11} + h_{22} + h_{33}) + \frac{1}{4}(h_{0110} + h_{2332} + h_{0330} + h_{1221} + h_{0220} - h_{0202} + h_{1331} - h_{1313})\right] \nonumber \\
&+ \mathbf{ZIII} \left[-\frac{1}{2}h_{00} - \frac{1}{4}(h_{0110} + h_{0330} + h_{0220} - h_{0202})\right] \nonumber \\
&+ \mathbf{IZII} \left[\frac{1}{4}h_{0110}\right] \nonumber \\
&+ \mathbf{IIZI} \left[-\frac{1}{2}h_{22} - \frac{1}{4}(h_{2332} + h_{1221} + h_{0220} - h_{0202})\right] \nonumber \\
&+ \mathbf{ZZII} \left[-\frac{1}{2}h_{11} - \frac{1}{4}(h_{0110} + h_{1221} + h_{1331} - h_{1313})\right] \nonumber \\
&+ \mathbf{ZIZI} \left[\frac{1}{4}(h_{0220} - h_{0202})\right] \nonumber \\
&+ \mathbf{IZIZ} \left[\frac{1}{4}h_{2332}\right] \nonumber \\
&+ \mathbf{XZXI} \left[\frac{1}{8}(h_{0132} + h_{0312})\right] \nonumber \\
&+ \mathbf{YZYI} \left[\frac{1}{8}(h_{0132} + h_{0312})\right] \nonumber \\
&+ \mathbf{ZZZI} \left[\frac{1}{4}h_{1221}\right] \nonumber \\
&+ \mathbf{ZIZZ} \left[\frac{1}{4}(h_{1331} - h_{1313})\right] \nonumber \\
&+ \mathbf{IZZZ} \left[-\frac{1}{2}h_{33} - \frac{1}{4}(h_{2332} + h_{0330} + h_{1331} - h_{1313})\right] \nonumber \\
&+ \mathbf{XZXZ} \left[\frac{1}{8}(h_{0132} + h_{0312})\right] \nonumber \\
&+ \mathbf{YZYZ} \left[\frac{1}{8}(h_{0132} + h_{0312})\right] \nonumber \\
&+ \mathbf{ZZZZ} \left[\frac{1}{4}h_{0330}\right]
\end{align}
$$


In [33]:
# Define the integrals
h00 = h11 = -1.252477
h22 = h33 = -0.475934
h0110 = 0.674493
h2332 = 0.697397
h0220 = h0330 = h1221 = h1331 = 0.663472
h0202 = h1313 = h0312 = h0132 = 0.181287

# Calculate coefficients for each Pauli string
coeffs = [
    # IIII
    0.5*(h00 + h11 + h22 + h33) + 0.25*(h0110 + h2332 + h0330 + h1221 + h0220 - h0202 + h1331 - h1313),
    # ZIII
    -0.5*h00 - 0.25*(h0110 + h0330 + h0220 - h0202),
    # IZII
    0.25*h0110,
    # IIZI
    -0.5*h22 - 0.25*(h2332 + h1221 + h0220 - h0202),
    # ZZII
    -0.5*h11 - 0.25*(h0110 + h1221 + h1331 - h1313),
    # ZIZI
    0.25*(h0220 - h0202),
    # IZIZ
    0.25*h2332,
    # XZXI
    0.125*(h0132 + h0312),
    # YZYI
    0.125*(h0132 + h0312),
    # ZZZI
    0.25*h1221,
    # ZIZZ
    0.25*(h1331 - h1313),
    # IZZZ
    -0.5*h33 - 0.25*(h2332 + h0330 + h1331 - h1313),
    # XZXZ
    0.125*(h0132 + h0312),
    # YZYZ
    0.125*(h0132 + h0312),
    # ZZZZ
    0.25*h0330
]

# Create the SparsePauliOp
H = SparsePauliOp(
    data=["IIII", 
          "ZIII", "IZII", "IIZI", 
          "ZZII", "ZIZI", "IZIZ", 
          "XZXI", "YZYI", 
          "ZZZI", "ZIZZ", "IZZZ", 
          "XZXZ", "YZYZ", 
          "ZZZZ"],
    coeffs=coeffs
)
# Convert the Hamiltonian to a matrix (assuming H is your SparsePauliOp)
H_matrix = H.to_matrix()

# Clean up near-zero values and convert to sympy Matrix
H_clean = np.real(H_matrix)  # Take real part since H is Hermitian
H_clean[np.abs(H_clean) < 1e-10] = 0  # Set very small values to exactly 0

# Convert to sympy Matrix with controlled precision
H_sympy = Matrix(H_clean)

# Round to reasonable precision (e.g., 6 decimal places)
H_rounded = H_sympy.applyfunc(lambda x: Float(x, 6) if x != 0 else 0)

# Generate LaTeX with better formatting
latex_str = latex(H_rounded)

# Display in Jupyter
display(Latex(f"$$H = {latex_str}$$"))

def format_matrix_latex(matrix, precision=17):
    """Format a numpy array as LaTeX matrix with controlled precision."""
    n = matrix.shape[0]
    
    # Start the LaTeX matrix
    latex_str = r"\begin{pmatrix}"
    
    for i in range(n):
        row_str = []
        for j in range(n):
            val = np.real(matrix[i, j])
            if np.abs(val) < 1e-18:
                row_str.append("0")
            elif np.abs(val - np.round(val)) < 1e-18:
                row_str.append(f"{int(np.round(val))}")
            else:
                row_str.append(f"{val:.{precision}f}")
        
        latex_str += " & ".join(row_str)
        if i < n - 1:
            latex_str += r" \\ "
    
    latex_str += r"\end{pmatrix}"
    return latex_str

# Use the custom formatter
latex_output = format_matrix_latex(H_matrix)
display(latex_output)
# display(Latex(f"$$H = {latex_output}$$"))

# display(Math(latex(Matrix(H))))
eigvals_bk_rodrigo, eigenvects = np.linalg.eig(H.to_matrix())
sorted(eigvals_bk_rodrigo)

<IPython.core.display.Latex object>

'\\begin{pmatrix}0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & -0.47593400000000019 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & -0.25447100000000000 & 0 & 0 & 0 & 0 & 0 & 0.18128700000000000 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & -0.47593400000000019 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 & -1.24622599999999961 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 & 0 & -1.25247700000000028 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 & 0 & 0 & -1.06493900000000008 & 0 & 0 & 0 & 0 & 0 & -0.18128700000000000 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 & 0 & 0 & 0 & -0.36129099999999992 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0.18128700000000000 & 0 & 0 & 0 & 0 & 0 & -1.83046100000000012 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & -1.16073800000000027 & 0 & 0 & 0 & 0 & 0 & 0 \\\\ 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0.20638199999999993 & 0 & 0 & 0 & 0 & 0 \\\\ 0 

[np.complex128(-1.8510456784448643+0j),
 np.complex128(-1.2524770000000003+0j),
 np.complex128(-1.2524770000000003+0j),
 np.complex128(-1.2462260000000003+0j),
 np.complex128(-1.246226+0j),
 np.complex128(-1.2462259999999996+0j),
 np.complex128(-1.1607380000000003+0j),
 np.complex128(-1.160738+0j),
 np.complex128(-0.8836520000000005+6.162975822039155e-33j),
 np.complex128(-0.4759340000000002+0j),
 np.complex128(-0.4759340000000002+0j),
 np.complex128(-0.36129100000000014+0j),
 np.complex128(-0.3612909999999999+0j),
 np.complex128(-0.23388632155513583+0j),
 np.complex128(-3.3306690738754696e-16+0j),
 np.complex128(0.20638199999999993+0j)]

## Just copying the paper's expression

In [28]:
H = SparsePauliOp(data=["IIII", 
                        "ZIII", "IZII", "IIZI", 
                        "ZZII", "ZIZI", "IZIZ", 
                        "XZXI", "YZYI", 
                        "ZZZI", "ZIZZ", "IZZZ", 
                        "XZXZ", "YZYZ", 
                        "ZZZZ"],
                  coeffs=[-0.81261, 
                          0.171201, 0.16862325, -0.2227965, 
                          0.171201, 0.12054625, 0.17434925, 
                          0.04532175, 0.04532175,
                          0.165868, 0.12054625, -0.2227965, 
                          0.04532175, 0.04532175,
                        0.165868])
display(Math(latex(Matrix(H))))
eigvals_bk, eigenvects = np.linalg.eig(H.to_matrix())
sorted(eigvals_bk)


<IPython.core.display.Math object>

[np.complex128(-1.8510456784448643+0j),
 np.complex128(-1.2524770000000003+0j),
 np.complex128(-1.2524770000000003+0j),
 np.complex128(-1.2462260000000005+0j),
 np.complex128(-1.246226+0j),
 np.complex128(-1.2462259999999996+0j),
 np.complex128(-1.1607379999999998+0j),
 np.complex128(-1.1607379999999998+0j),
 np.complex128(-0.8836520000000003+0j),
 np.complex128(-0.4759340000000002+0j),
 np.complex128(-0.4759340000000002+0j),
 np.complex128(-0.3612909999999999+0j),
 np.complex128(-0.3612909999999999+0j),
 np.complex128(-0.23388632155513583+0j),
 np.complex128(-3.0531133177191805e-16+0j),
 np.complex128(0.20638200000000012+0j)]

In [30]:
for i in range(len(eigvals_bk_rodrigo)):
    print(f"{eigvals_bk_rodrigo[i]} {'=' if eigvals_bk_rodrigo[i] == eigvals_bk[i] else '!=' } {eigvals_bk[i]}")

(-1.246226+0j) != (-1.2462260000000005+0j)
(-0.8836520000000005+6.162975822039155e-33j) != (-0.8836520000000003+0j)
(-0.23388632155513583+0j) = (-0.23388632155513583+0j)
(-1.8510456784448643+0j) = (-1.8510456784448643+0j)
(-3.3306690738754696e-16+0j) != (-3.0531133177191805e-16+0j)
(-0.4759340000000002+0j) = (-0.4759340000000002+0j)
(-0.4759340000000002+0j) = (-0.4759340000000002+0j)
(-1.2462259999999996+0j) = (-1.2462259999999996+0j)
(-1.2524770000000003+0j) = (-1.2524770000000003+0j)
(-0.3612909999999999+0j) = (-0.3612909999999999+0j)
(-1.1607380000000003+0j) != (-1.1607379999999998+0j)
(0.20638199999999993+0j) != (0.20638200000000012+0j)
(-1.160738+0j) != (-1.1607379999999998+0j)
(-1.2524770000000003+0j) = (-1.2524770000000003+0j)
(-1.2462260000000003+0j) != (-1.246226+0j)
(-0.36129100000000014+0j) != (-0.3612909999999999+0j)


In [20]:
H_JW = SparsePauliOp(data=["IIII", 
                           "ZIII", "IZII", "IIZI", "IIIZ",
                           "ZZII", "ZIZI", "IZZI", "ZIIZ", "IZIZ", "IIZZ",
                           "YYXX", "XYYX", "YXXY", "XXYY"],
                           coeffs=[-0.81261,
                                   0.171201,  0.171201, -0.2227965, -0.2227965,
                                   0.16862325,  0.12054625, 0.165868, 0.165868, 0.12054625, 0.17434925, 
                                   -0.04532175, 0.04532175, 0.04532175, -0.04532175])

display(Math(latex(Matrix(H_JW))))
eigvals_jw, eigenvects = np.linalg.eig(H_JW.to_matrix())
sorted(eigvals_jw)


<IPython.core.display.Math object>

[np.complex128(-1.8510456784448643+0j),
 np.complex128(-1.2524770000000003+0j),
 np.complex128(-1.2524770000000003+0j),
 np.complex128(-1.246226-2.311115933264683e-33j),
 np.complex128(-1.246226+0j),
 np.complex128(-1.246226+0j),
 np.complex128(-1.160738+0j),
 np.complex128(-1.1607379999999998+0j),
 np.complex128(-0.8836520000000005+0j),
 np.complex128(-0.4759340000000002+0j),
 np.complex128(-0.47593400000000013+0j),
 np.complex128(-0.3612909999999999+0j),
 np.complex128(-0.3612909999999999+0j),
 np.complex128(-0.23388632155513578+0j),
 np.complex128(-8.326672684688674e-17+0j),
 np.complex128(0.20638200000000018+0j)]

In [21]:
eigvals_jw == eigvals_bk

array([False, False, False, False, False, False,  True, False, False,
        True, False, False, False, False, False, False])